## CONFIG AND SETUP

In [ ]:
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 15
%region us-east-1
# %iam_role arn:aws:iam::958165011713:role/AWSGlueServiceRole-IcebergLab

: 

In [ ]:
%%configure
{
  "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.warehouse=s3://iceberg-data-lake-958165011713/glue-iceberg-warehouse/ --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO"
}

In [ ]:
spark
spark.sql("SELECT 1 AS test").show()

In [ ]:
spark.sql("SHOW DATABASES IN glue_catalog").show() #####checking our database is present or not

In [ ]:
#######################CHECK SPARK S3 CONNECTIVITY
spark.sql("""
CREATE TABLE IF NOT EXISTS glue_catalog.practice.spark_iceberg_test (
    id BIGINT,
    name STRING,
    city STRING,
    created_at TIMESTAMP
)
USING iceberg
""")

spark.sql("""
INSERT INTO glue_catalog.practice.spark_iceberg_test VALUES
(1, 'Ayush', 'Kolkata', current_timestamp()),
(2, 'Rahul', 'Delhi', current_timestamp()),
(3, 'Sneha', 'Bangalore', current_timestamp())
""")

spark.sql("""
SELECT *
FROM glue_catalog.practice.spark_iceberg_test
""").show()

## REAL WORK

In [ ]:
## Read already created dataset (Change your account name below)
orders_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("s3://iceberg-data-lake-958165011713/raw/500_k/orders_500k.csv")
)

In [ ]:
orders_df.printSchema()

In [ ]:
orders_df.show(5)

In [ ]:
#Register temp view:
orders_df.createOrReplaceTempView("orders_raw")
spark.sql("SHOW TABLES IN practice").show(truncate=False)

In [ ]:
spark.sql("""
CREATE TABLE glue_catalog.practice.orders_spark_month_city_customer_bucket
USING iceberg
PARTITIONED BY (
  months(order_date),
  city,
  bucket(10, customer_id)
)
AS
SELECT
  CAST(order_id AS BIGINT) AS order_id,
  CAST(customer_id AS STRING) AS customer_id,
  CAST(order_date AS DATE) AS order_date,
  product_category,
  city,
  CAST(order_amount AS DOUBLE) AS order_amount,
  order_status,
  payment_status,
  current_timestamp() AS updated_at
FROM orders_raw
WHERE customer_id IS NOT NULL
""")

In [ ]:
spark.sql("""
SELECT COUNT(*) AS total_partitions
FROM glue_catalog.practice.orders_spark_month_city_customer_bucket.partitions
""").show()

## PYSPARK OPTIMIZATION - PART 1

### Broadcast

In [ ]:
orders_df = spark.read.table("glue_catalog.practice.orders_500k_iceberg_partitionby_month")
orders_df.printSchema()
print(orders_df.count())
orders_df.rdd.getNumPartitions()

In [ ]:
#Creating a small dimension table
city_dim_raw = [
    ("Kolkata", "Tier 1", "East"),
    ("Delhi", "Tier 1", "North"),
    ("Mumbai", "Tier 1", "West"),
    ("Bangalore", "Tier 1", "South"),
    ("Chennai", "Tier 1", "South"),
    ("Hyderabad", "Tier 1", "South"),
    ("Pune", "Tier 1", "West"),
    ("Ahmedabad", "Tier 2", "West"),
    ("Jaipur", "Tier 2", "North"),
    ("Lucknow", "Tier 2", "North")
]

city_dim = spark.createDataFrame(
    city_dim_raw,
    ["city", "city_tier", "region"]
)

city_dim.show()

In [ ]:
#DISABLING AUTOMATIC BROADCAST
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [ ]:
#JOIN WITHOUT BROADCAST
normal_join_df = (
    orders_df
    .join(city_dim, on="city", how="left")
    .groupBy("region", "city_tier")
    .count()
)
normal_join_df.explain("formatted")

In [ ]:
"""
Read orders Iceberg table
Read small city_dim
Shuffle both sides by city
Sort both sides by city
Perform SortMergeJoin
Group by region, city_tier
Shuffle again for final aggregation
Return result

The important thing:

Spark used SortMergeJoin, not BroadcastHashJoin.
"""

In [ ]:
#PERFORMANCE WITHOUT BROADCAST
import time
start = time.time()
normal_join_df.show()
end = time.time()
print(f"Normal join runtime: {round(end - start, 2)} seconds")

In [ ]:
#JOIN BY BROADCAST
from pyspark.sql.functions import broadcast
broadcast_join_df = (
    orders_df
    .join(broadcast(city_dim), on="city", how="left")
    .groupBy("region", "city_tier")
    .count()
)
broadcast_join_df.explain("formatted")

In [ ]:
start = time.time()

broadcast_join_df.show()

end = time.time()
print(f"Broadcast join runtime: {round(end - start, 2)} seconds")

In [ ]:
#ENABLING BROADCAST THRESHOLD SETTING IT TO 10 MB
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

### Shuffle Optimization

In [ ]:
#BAD SHUFFLE
bad_shuffle_df = (
    orders_df
    .groupBy("city")
    .count()
)
bad_shuffle_df.explain("formatted")

In [ ]:
#PERFORMANCE WITH BAD SHUFFLE
import time

start = time.time()
bad_shuffle_df.show()
end = time.time()

print(f"Bad shuffle runtime: {round(end - start, 2)} seconds")

In [ ]:
#GOOD SHUFFLE
good_shuffle_df = (
    orders_df
    .filter("order_date >= DATE '2026-01-01' AND order_date < DATE '2026-02-01'")
    .select("city")
    .groupBy("city")
    .count()
)

good_shuffle_df.explain("formatted")

In [ ]:
#GOOD SHUFFLE PERFORMANCE
start = time.time()
good_shuffle_df.show()
end = time.time()

print(f"Good shuffle runtime: {round(end - start, 2)} seconds")

### AQE

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "100")

In [ ]:
#WITHOUT AQE
no_aqe_df = (
    orders_df
    .filter("order_date >= DATE '2026-01-01' AND order_date < DATE '2026-02-01'")
    .groupBy("city")
    .count()
)

no_aqe_df.explain("formatted")

In [ ]:
#PERFORMANCE BEFORE AQE
import time

start = time.time()
no_aqe_df.show()
end = time.time()

print(f"Without AQE runtime: {round(end - start, 2)} seconds")

In [ ]:
#ENABLING AQE
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "100")

In [ ]:
#SAME QUERY
aqe_df = (
    orders_df
    .filter("order_date >= DATE '2026-01-01' AND order_date < DATE '2026-02-01'")
    .groupBy("city")
    .count()
)

aqe_df.explain("formatted")

In [ ]:
#PERFORMANCE AFTER AQE
start = time.time()
aqe_df.show()
end = time.time()

print(f"With AQE runtime: {round(end - start, 2)} seconds")

### Memory Optimization

In [ ]:
#WITHOUT CACHE
jan_orders_df = (
    orders_df
    .filter("order_date >= DATE '2026-01-01' AND order_date < DATE '2026-02-01'")
    .select("order_id", "city", "product_category", "order_amount")
)

import time

start = time.time()

jan_orders_df.groupBy("city").count().show()
jan_orders_df.groupBy("product_category").count().show()
jan_orders_df.agg({"order_amount": "sum"}).show()

end = time.time()

print(f"Without cache runtime: {round(end - start, 2)} seconds")

In [ ]:
# WITH CACHE
jan_orders_cached_df = jan_orders_df.cache()
jan_orders_cached_df.count()

start = time.time()

jan_orders_cached_df.groupBy("city").count().show()
jan_orders_cached_df.groupBy("product_category").count().show()
jan_orders_cached_df.agg({"order_amount": "sum"}).show()

end = time.time()

print(f"With cache runtime: {round(end - start, 2)} seconds")

In [ ]:
jan_orders_cached_df.unpersist()

In [ ]:
Situation 1: Working with a Cached Dataframe
CPU: x1
Memory: y1

Situation 2: Working with a non Cached Dataframe
CPU: x2
Memory: y2


CPU Usage
1) x1 > x2 
2) x1 == x2
3) x1 < x2 

Memory Usage
1) y1 > y2
2) y1 == y2 
3) y1 < y2 

Answer:
For first execution only:

CPU: x1 ≈ x2
Memory during computation: y1 ≈ y2
Memory after computation: y1 > y2

For repeated usage:

CPU: x1 < x2
Memory: y1 > y2